# Diabetes Prediction with Logistic Regression

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded.')

## Exercise 1: Understanding the Problem and Data Collection

In [ ]:
# ── Load the dataset ──────────────────────────────────────────
URL = ('https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/'
       'Week%204/Day%202/Diabetes%20prediction%20dataset.zip')

try:
    r = requests.get(URL, timeout=15)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_file = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_file) as f:
            df = pd.read_csv(f)
    print('Loaded from URL.')
except Exception as e:
    print(f'URL failed ({e}). Generating representative synthetic dataset...')
    np.random.seed(42)
    n = 100_000
    gender   = np.random.choice(['Male','Female','Other'], n, p=[0.41,0.58,0.01])
    age      = np.round(np.random.uniform(0.08, 80, n), 2)
    hypert   = np.random.choice([0,1], n, p=[0.925, 0.075])
    heart    = np.random.choice([0,1], n, p=[0.960, 0.040])
    smoking  = np.random.choice(
        ['never','No Info','current','former','ever','not current'],
        n, p=[0.35,0.35,0.09,0.09,0.06,0.06])
    bmi          = np.round(np.random.normal(27.3, 6.6, n).clip(10.0, 96.0), 2)
    hba1c        = np.round(np.random.choice(np.arange(3.5, 9.1, 0.1), n), 1)
    blood_glucose = np.random.choice(
        [80,85,90,100,126,140,158,200,260,300], n,
        p=[0.15,0.15,0.15,0.15,0.10,0.10,0.08,0.06,0.04,0.02])
    prob = (0.02
            + 0.06*(hba1c > 6.5)
            + 0.05*(blood_glucose > 125)
            + 0.03*(age > 50)
            + 0.02*(bmi > 30)
            + 0.03*hypert
            + 0.02*heart).clip(0, 1)
    diabetes = (np.random.rand(n) < prob).astype(int)
    df = pd.DataFrame({
        'gender':gender, 'age':age, 'hypertension':hypert,
        'heart_disease':heart, 'smoking_history':smoking,
        'bmi':bmi, 'HbA1c_level':hba1c,
        'blood_glucose_level':blood_glucose, 'diabetes':diabetes
    })
    print('Synthetic dataset generated.')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Exploration ───────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Statistics ===')
df.describe().round(2)

In [ ]:
# ── Class distribution ────────────────────────────────────────
counts = df['diabetes'].value_counts()
pct    = df['diabetes'].value_counts(normalize=True) * 100

print('Positive cases (diabetes = 1):', counts[1],
      f'({pct[1]:.1f}%)')
print('Negative cases (diabetes = 0):', counts[0],
      f'({pct[0]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Diabetes (0)','Diabetes (1)'], counts.values,
            color=['#4C72B0','#E84040'], edgecolor='white', width=0.5)
for i, (v, p) in enumerate(zip(counts.values, pct.values)):
    axes[0].text(i, v + 500, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Class Distribution — Diabetes', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(counts.values, labels=['No Diabetes','Diabetes'],
            autopct='%1.1f%%', colors=['#4C72B0','#E84040'],
            startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature distributions by class ───────────────────────────
num_cols = ['age','bmi','HbA1c_level','blood_glucose_level']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.flatten(), num_cols):
    for label, color in [(0,'#4C72B0'),(1,'#E84040')]:
        ax.hist(df[df['diabetes']==label][col],
                bins=40, alpha=0.55, color=color, edgecolor='white',
                label=f'Diabetes={label}')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distributions by Diabetes Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train / Test split ────────────────────────────────────────
# Encode categoricals
df_enc = df.copy()
le = LabelEncoder()
df_enc['gender']          = le.fit_transform(df_enc['gender'])
df_enc['smoking_history'] = le.fit_transform(df_enc['smoking_history'])

X = df_enc.drop(columns='diabetes')
y = df_enc['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]:,} rows  ({y_train.mean():.1%} positive)')
print(f'Test set     : {X_test.shape[0]:,} rows  ({y_test.mean():.1%} positive)')

## Exercise 2: Model Picking and Standardization

### Which classification model should we use?

**Logistic Regression** is the natural first choice for this binary classification problem because:
- The target variable is binary (0 = no diabetes, 1 = diabetes).
- It outputs a calibrated probability between 0 and 1, which maps directly to the question "how likely is this patient to have diabetes?".
- It is interpretable — each coefficient shows the direction and magnitude of a feature's effect on the log-odds of diabetes.
- It serves as a strong, well-understood baseline before trying more complex models (Random Forest, XGBoost, Neural Network).

### Do we need to standardize the data?

**Yes.** Logistic Regression uses gradient-based optimization. Features on very different scales (e.g., `age` 0–80 vs `HbA1c_level` 3.5–9.0 vs `blood_glucose_level` 80–300) cause the optimization to converge slowly and can give disproportionate weight to features with large numerical ranges. `StandardScaler` transforms each feature to zero mean and unit variance, ensuring fair and fast convergence.

In [ ]:
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on train only
X_test_s  = scaler.transform(X_test)        # apply same transform to test

# Verify scaling
print('After scaling — train set statistics (should be ≈ mean=0, std=1):')
print(pd.DataFrame(X_train_s, columns=X.columns).describe().round(3).loc[['mean','std']])

## Exercise 3: Model Training

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_s, y_train)

print('Logistic Regression trained successfully.')
print(f'Iterations run: {model.n_iter_[0]}')
print('\nCoefficients (feature → log-odds of diabetes):')
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_[0]})\
            .sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

In [ ]:
# Coefficient bar chart
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E84040' if v > 0 else '#4C72B0' for v in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(positive = increases diabetes risk)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient (log-odds)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 4: Evaluation Metrics

In [ ]:
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

acc = accuracy_score(y_test, y_pred)
print(f'Accuracy : {acc:.4f}  ({acc*100:.2f}%)')

**Accuracy comment:**  
The accuracy score represents the percentage of all predictions (both positive and negative) that are correct. However, because the dataset is **imbalanced** (~91% negative cases), a naive model that always predicts "no diabetes" would also achieve ~91% accuracy. We therefore rely heavily on precision, recall, and F1-score — and `class_weight='balanced'` was used during training to compensate for this imbalance.

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw counts
disp = ConfusionMatrixDisplay(cm, display_labels=['No Diabetes','Diabetes'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

# Normalised
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2   = ConfusionMatrixDisplay(cm_norm.round(3), display_labels=['No Diabetes','Diabetes'])
disp2.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalised)', fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correct No-Diabetes) : {tn:,}')
print(f'False Positives (No-Diabetes → Diabetes): {fp:,}')
print(f'False Negatives (Diabetes → No-Diabetes): {fn:,}')
print(f'True Positives  (correct Diabetes)     : {tp:,}')

**Confusion matrix comment:**  
- **True Negatives (TN)**: Correctly identified as non-diabetic — the majority class, so naturally high.
- **True Positives (TP)**: Correctly identified as diabetic — the most important group for clinical purposes.
- **False Negatives (FN)**: Diabetic patients incorrectly predicted as non-diabetic. In healthcare, this is the most dangerous error (missed diagnoses), so minimising FN is critical.
- **False Positives (FP)**: Non-diabetic patients incorrectly flagged as diabetic. Less dangerous but leads to unnecessary follow-up tests.

In [ ]:
# ── Precision, Recall, F1 ─────────────────────────────────────
report = classification_report(y_test, y_pred,
                                target_names=['No Diabetes','Diabetes'],
                                output_dict=True)
print(classification_report(y_test, y_pred,
                             target_names=['No Diabetes','Diabetes']))

In [ ]:
# Visual comparison
metrics_data = {
    'Precision': [report['No Diabetes']['precision'], report['Diabetes']['precision']],
    'Recall':    [report['No Diabetes']['recall'],    report['Diabetes']['recall']],
    'F1-score':  [report['No Diabetes']['f1-score'],  report['Diabetes']['f1-score']],
}
x      = np.arange(2)
labels = ['No Diabetes', 'Diabetes']
width  = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
colors_m = ['#4C72B0', '#55A868', '#DD8452']
for i, (metric, vals) in enumerate(metrics_data.items()):
    bars = ax.bar(x + i*width, vals, width, label=metric,
                  color=colors_m[i], edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Precision, Recall and F1-Score by Class', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Precision, Recall, F1 comments:**

- **Precision (Diabetes class)**: Of all patients predicted diabetic, the proportion that truly are diabetic. A low value means many false alarms.
- **Recall (Diabetes class)**: Of all actual diabetic patients, the proportion our model detected. This is the critical metric in healthcare — a low recall means many patients are missed.
- **F1-score**: The harmonic mean of precision and recall. Useful when the classes are imbalanced, as it penalises models that sacrifice one for the other.
- Using `class_weight='balanced'` increases recall for the minority (diabetes) class at the cost of slightly lower precision — an acceptable trade-off in a medical screening context where false negatives are more costly than false positives.

## Exercise 5: Visualizing the Decision Boundary

In [ ]:
# We reduce to 2D with PCA to be able to plot a 2D decision boundary
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_s)
X_test_2d  = pca.transform(X_test_s)

# Retrain LR on the 2D representation
lr_2d = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_2d.fit(X_train_2d, y_train)
acc_2d = accuracy_score(y_test, lr_2d.predict(X_test_2d))

# Mesh grid
h     = 0.15
x_min = X_test_2d[:,0].min() - 1
x_max = X_test_2d[:,0].max() + 1
y_min = X_test_2d[:,1].min() - 1
y_max = X_test_2d[:,1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = lr_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
ax.contour( xx, yy, Z, colors='black', linewidths=1, linestyles='--')

# Scatter test points (sample for clarity)
idx_sample = np.random.choice(len(X_test_2d), size=min(2000, len(X_test_2d)), replace=False)
scatter = ax.scatter(
    X_test_2d[idx_sample, 0], X_test_2d[idx_sample, 1],
    c=y_test.values[idx_sample], cmap='RdBu',
    edgecolors='white', linewidths=0.4, s=25, alpha=0.7
)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#3D85C8', label='No Diabetes (0)'),
                   Patch(color='#C44E52', label='Diabetes (1)')],
          title='True Label', fontsize=9)

ax.set_title(f'Decision Boundary (PCA 2D projection)\nAccuracy on 2D: {acc_2d:.3f}',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Full model accuracy (all features): {accuracy_score(y_test, y_pred):.4f}')
print(f'2D PCA model accuracy             : {acc_2d:.4f}')
print(f'Variance explained by PC1+PC2     : {pca.explained_variance_ratio_.sum()*100:.1f}%')

## Exercise 6: ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

# ── ROC plot following the statology.org template ─────────────
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr, tpr, color='blue', linewidth=2.5,
        label=f'Logistic Regression (AUC = {auc_score:.3f})')
ax.plot([0, 1], [0, 1], color='darkgrey', linestyle='--',
        linewidth=1.5, label='Random Classifier (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.10, color='blue')

# Mark the optimal threshold (closest to top-left corner)
optimal_idx = np.argmax(tpr - fpr)
ax.scatter(fpr[optimal_idx], tpr[optimal_idx],
           color='red', s=100, zorder=5,
           label=f'Optimal threshold = {thresholds[optimal_idx]:.2f}')

ax.set_title('ROC Curve — Logistic Regression (Diabetes Prediction)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC-ROC Score : {auc_score:.4f}')

In [ ]:
# ── Threshold sensitivity analysis ───────────────────────────
precisions, recalls, f1s, thres_vals = [], [], [], []
from sklearn.metrics import precision_score, recall_score, f1_score

for t in np.arange(0.1, 0.9, 0.05):
    yp_t = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, yp_t, zero_division=0))
    recalls.append(recall_score(y_test, yp_t))
    f1s.append(f1_score(y_test, yp_t, zero_division=0))
    thres_vals.append(t)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thres_vals, precisions, label='Precision', color='#4C72B0', linewidth=2)
ax.plot(thres_vals, recalls,    label='Recall',    color='#E84040', linewidth=2)
ax.plot(thres_vals, f1s,        label='F1-score',  color='#55A868', linewidth=2)
ax.axvline(thresholds[optimal_idx], color='grey', linestyle='--', linewidth=1.5,
           label=f'Optimal threshold ({thresholds[optimal_idx]:.2f})')
ax.set_title('Precision / Recall / F1 vs Classification Threshold',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.legend(fontsize=9)
ax.set_xlim(0.1, 0.85)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**ROC Curve comments:**

- The **ROC curve** plots the True Positive Rate (Recall) against the False Positive Rate at every possible classification threshold.
- The **AUC (Area Under the Curve)** summarises discriminative power in a single number: 0.5 = random guessing, 1.0 = perfect classifier.
- An AUC above 0.80 indicates that our model has strong discriminative ability — it ranks a randomly chosen diabetic patient above a non-diabetic patient ~80%+ of the time.
- The **optimal threshold** (red dot) is the point on the curve closest to the top-left corner, maximising `TPR - FPR`. In a clinical context, a doctor may choose to **lower the threshold** below 0.5 to increase recall (catch more actual diabetics) at the cost of higher false positive rate.
- The threshold sensitivity chart confirms this trade-off: as the threshold decreases, recall rises while precision falls.